<a href="https://colab.research.google.com/github/louistrue/learn-ifc-bfh25-D/blob/main/MVP/NPK_IFC_Mapper_louistrue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NPK 241 - IFC Quantity Takeoff

This notebook extracts structural elements from IFC models and maps them to **NPK 241 (Ortbetonbau)** catalog positions for cost estimation.

**What you'll learn:**
- Extract walls and slabs from IFC models with their quantities
- Map concrete types and dimensions to NPK position codes
- Generate both concrete (631.xxx) and formwork (231.xxx) positions
- Export professional Excel reports with dynamic subtotals

**NPK 241 Structure:**
- `61.xxx` - Concrete grades (NPK A, B, C, D...)
- `231.xxx` - Wall formwork (Schalung) by height
- `631.xxx` - Wall concrete by height and thickness
- `261.xxx` - Slab formwork by thickness
- `613.xxx` - Slab concrete by thickness

## 1. Install Dependencies

Run this cell to install required Python packages.

In [ ]:
%pip install ifcopenshell pandas openpyxl ipywidgets

## 2. Configuration - NPK Mappings

These mappings define how IFC properties translate to NPK positions.
Edit these if your project uses different Betontyp codes.

In [ ]:
from dataclasses import dataclass
from typing import Optional, List, Tuple

# =============================================================================
# NPK 241 CONFIGURATION
# =============================================================================

@dataclass
class DimensionCategory:
    """Represents an NPK height/thickness category."""
    name: str
    min_val: float
    max_val: float
    code: int

# Concrete type mapping: IFC Betontyp -> (NPK Label, Position Code)
# Adjust these based on your project's property definitions!
BETONTYP_TO_NPK = {
    "01": ("NPK C", 300),  # Special concrete
    "02": ("NPK B", 200),  # Increased requirements
    "03": ("NPK A", 100),  # Standard concrete
    "04": ("NPK B", 200),  # Also NPK B
}

# Wall height categories per NPK 241 (positions 231 & 631)
WALL_HEIGHT_CATEGORIES = [
    DimensionCategory("bis 1.5m", 0.0, 1.50, 10),
    DimensionCategory("1.51-1.99m", 1.51, 1.99, 20),
    DimensionCategory("2.0-2.99m", 2.00, 2.99, 30),
    DimensionCategory("3.0-4.0m", 3.00, 4.00, 40),
]

# Slab thickness categories per NPK 241 (positions 261 & 613)
SLAB_THICKNESS_CATEGORIES = [
    DimensionCategory("bis 0.15m", 0.0, 0.15, 10),
    DimensionCategory("0.15-0.25m", 0.15, 0.25, 20),
    DimensionCategory("0.25-0.35m", 0.25, 0.35, 30),
    DimensionCategory("0.35-0.45m", 0.35, 0.45, 40),
    DimensionCategory(">0.45m", 0.45, 10.0, 50),
]

# Property set names to search for (different authoring tools use different names)
PSET_NAMES = {
    "custom": ["wg_SUB", "Pset_WallCommon", "ArchiCADProperties"],
    "betontyp_props": ["Betontyp", "ConcreteType", "Betongüte"],
}

print("✅ NPK configuration loaded")
print(f"   Concrete types: {list(BETONTYP_TO_NPK.keys())}")
print(f"   Wall height ranges: {[c.name for c in WALL_HEIGHT_CATEGORIES]}")
print(f"   Slab thickness ranges: {[c.name for c in SLAB_THICKNESS_CATEGORIES]}")

## 3. NPK Position Generation Functions

Core logic for generating complete NPK position codes.

**NPK 631 Structure (Wall Concrete):**
- `631.1xx` = NPK A concrete
- `631.2xx` = NPK B concrete
- `631.3xx` = NPK C concrete
- `631.x1x` = Height bis 1.5m
- `631.x2x` = Height 1.51-1.99m
- `631.x3x` = Height 2.0-2.99m
- `631.x4x` = Height 3.0-4.0m
- `631.xx4` = With thickness (Wanddicke)

In [ ]:
def find_category(value: float, categories: List[DimensionCategory]) -> Optional[DimensionCategory]:
    """Find the matching category for a dimension value."""
    EPSILON = 0.001
    for cat in categories:
        if cat.min_val - EPSILON <= value <= cat.max_val + EPSILON:
            return cat
    return None


def generate_wall_positions(betontyp: str, height: float) -> dict:
    """
    Generate NPK positions for a wall element.
    
    Returns dict with:
    - npk_concrete: Position 631.xxx for concrete
    - npk_formwork: Position 231.xxx for formwork
    - npk_label: Human-readable NPK type (A, B, C)
    - height_category: Height range name
    - notes: Any warnings or special cases
    """
    result = {
        "npk_concrete": None,
        "npk_formwork": None,
        "npk_label": None,
        "height_category": None,
        "notes": None
    }
    
    # Get concrete type
    npk_info = BETONTYP_TO_NPK.get(str(betontyp))
    if npk_info is None:
        result["notes"] = f"Unknown Betontyp '{betontyp}'"
        return result
    
    npk_label, npk_code = npk_info
    result["npk_label"] = npk_label
    
    # Get height category
    height_cat = find_category(height, WALL_HEIGHT_CATEGORIES)
    if height_cat is None:
        if height > 4.0:
            result["notes"] = f"Height {height:.2f}m > 4.0m - requires Aufmass"
            result["npk_concrete"] = "AUFMASS"
            result["npk_formwork"] = "AUFMASS"
            result["height_category"] = ">4.0m"
        else:
            result["notes"] = f"Height {height:.2f}m outside standard ranges"
        return result
    
    result["height_category"] = height_cat.name
    
    # Generate position codes
    # Concrete: 631.[type+height+4] e.g., 631.134 = NPK A, 2.0-2.99m, with thickness
    result["npk_concrete"] = f"631.{npk_code + height_cat.code + 4}"
    
    # Formwork: 231.10[height_digit] e.g., 231.103 = doppelhäuptig, 2.0-2.99m
    result["npk_formwork"] = f"231.10{height_cat.code // 10}"
    
    return result


def generate_slab_positions(betontyp: str, thickness: float) -> dict:
    """
    Generate NPK positions for a slab element.
    
    Returns dict with:
    - npk_concrete: Position 613.xxx for concrete
    - npk_formwork: Position 261.xxx for formwork
    - npk_label: Human-readable NPK type
    - thickness_category: Thickness range name
    - notes: Any warnings
    """
    result = {
        "npk_concrete": None,
        "npk_formwork": None,
        "npk_label": None,
        "thickness_category": None,
        "notes": None
    }
    
    # Get concrete type
    npk_info = BETONTYP_TO_NPK.get(str(betontyp))
    if npk_info is None:
        result["notes"] = f"Unknown Betontyp '{betontyp}'"
        return result
    
    npk_label, npk_code = npk_info
    result["npk_label"] = npk_label
    
    # Get thickness category
    thick_cat = find_category(thickness, SLAB_THICKNESS_CATEGORIES)
    if thick_cat is None:
        result["notes"] = f"Thickness {thickness:.2f}m outside ranges"
        return result
    
    result["thickness_category"] = thick_cat.name
    
    # Generate position codes (simplified)
    result["npk_concrete"] = f"613.{npk_code // 100}{thick_cat.code // 10}1"
    result["npk_formwork"] = f"261.11{thick_cat.code // 10}"
    
    return result


# Quick test
print("Test wall position generation:")
test = generate_wall_positions("03", 2.5)
print(f"  Betontyp '03', Height 2.5m -> {test}")

test2 = generate_wall_positions("01", 5.0)
print(f"  Betontyp '01', Height 5.0m -> {test2}")

## 4. Load IFC Model

Choose how to load your IFC model:
- **Option A**: Download from GitHub repository
- **Option B**: Upload your own file

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import ifcopenshell
    from ifcopenshell.util.element import get_psets
except ImportError as exc:
    raise ImportError("ifcopenshell required. Run: pip install ifcopenshell") from exc

# Global state
model = None
df_walls = None
df_slabs = None

# GitHub repository details
GITHUB_REPO_OWNER = 'louistrue'
GITHUB_REPO_NAME = 'learn-ifc-bfh25-D'
GITHUB_FOLDER_PATH = 'Modelle/BFH-25'


def fetch_github_files():
    """Fetch list of IFC files from GitHub repository."""
    import urllib.request
    import json
    
    url = f'https://api.github.com/repos/{GITHUB_REPO_OWNER}/{GITHUB_REPO_NAME}/contents/{GITHUB_FOLDER_PATH}'
    try:
        with urllib.request.urlopen(url) as response:
            files_data = json.loads(response.read().decode())
            return [f['name'] for f in files_data if f['name'].endswith('.ifc')]
    except Exception as e:
        print(f"⚠️ Could not fetch from GitHub: {e}")
        return ['Error fetching files']


def extract_betontyp(psets: dict) -> Optional[str]:
    """Extract Betontyp from property sets, trying multiple locations."""
    for pset_name in PSET_NAMES["custom"]:
        pset = psets.get(pset_name, {})
        for prop_name in PSET_NAMES["betontyp_props"]:
            if prop_name in pset:
                betontyp = str(pset[prop_name]).strip()
                # Filter out invalid values (only accept valid NPK codes)
                if betontyp and betontyp in ["01", "02", "03", "04"]:
                    return betontyp
    return None


def extract_thickness_from_name(name: str) -> Optional[float]:
    """Extract thickness from element name (e.g., 'STB 28cm' -> 0.28)."""
    if not name:
        return None
    
    import re
    # Look for patterns like "28cm", "30cm", "120cm" etc.
    match = re.search(r'(\d+(?:\.\d+)?)\s*cm', name, re.IGNORECASE)
    if match:
        thickness_cm = float(match.group(1))
        return thickness_cm / 100.0  # Convert to meters
    
    return None


def extract_thickness_from_wg_sub(psets: dict) -> Optional[float]:
    """
    Extract thickness from wg_SUB property names/values.
    Looks for patterns like "d=65", "d=65cm" in property values.
    Example: "Bodenplatte Stahlbeton d=65 wasserdicht" -> 0.65m
    """
    wg_sub = psets.get("wg_SUB", {})
    
    # Check all properties in wg_SUB for thickness patterns
    for prop_name, prop_value in wg_sub.items():
        if prop_value:
            value_str = str(prop_value)
            # Look for patterns like "d=65", "d=65cm", "d=28", "d=28cm"
            import re
            match = re.search(r'd\s*=\s*(\d+(?:\.\d+)?)\s*(?:cm)?', value_str, re.IGNORECASE)
            if match:
                thickness_cm = float(match.group(1))
                return thickness_cm / 100.0  # Convert to meters
    
    return None


def extract_slab_thickness(slab, psets: dict, qto: dict) -> Optional[float]:
    """
    Extract slab thickness using multiple methods (in priority order):
    1. From wg_SUB properties (e.g., "d=65" in property values) - MOST RELIABLE
    2. From QTO Depth/Width
    3. From element name (e.g., "STB 28cm" -> 0.28m)
    4. Calculated from Volume/Area
    """
    # Method 1: From wg_SUB properties (PRIORITY)
    thickness = extract_thickness_from_wg_sub(psets)
    if thickness:
        return thickness
    
    # Method 2: Direct from QTO
    thickness = qto.get("Depth") or qto.get("Width")
    if thickness:
        return thickness
    
    # Method 3: From name
    name = slab.Name or ""
    thickness = extract_thickness_from_name(name)
    if thickness:
        return thickness
    
    # Method 4: Calculate from Volume/Area
    net_volume = qto.get("NetVolume")
    gross_area = qto.get("GrossArea")
    if net_volume and gross_area and gross_area > 0:
        calculated_thickness = net_volume / gross_area
        # Sanity check: reasonable thickness range
        if 0.05 <= calculated_thickness <= 2.0:
            return calculated_thickness
    
    return None


def load_and_process_ifc(file_path: str):
    """Load IFC and extract walls/slabs with NPK positions."""
    global model, df_walls, df_slabs
    
    print(f"📂 Loading {file_path}...")
    model = ifcopenshell.open(file_path)
    
    # =========== EXTRACT WALLS ===========
    walls = model.by_type("IfcWall") + model.by_type("IfcWallStandardCase")
    wall_rows = []
    
    for wall in walls:
        psets = get_psets(wall)
        qto = psets.get("Qto_WallBaseQuantities", {})
        
        betontyp = extract_betontyp(psets)
        height = qto.get("Height")
        
        row = {
            "GlobalId": wall.GlobalId,
            "Name": wall.Name or "",
            "Betontyp": betontyp,
            "Height_m": height,
            "Width_m": qto.get("Width"),
            "NetVolume_m3": qto.get("NetVolume"),
            "NetSideArea_m2": qto.get("NetSideArea"),
        }
        
        # Generate NPK positions if we have required data
        if betontyp and height:
            npk = generate_wall_positions(betontyp, height)
            row.update({
                "NPK_Concrete": npk["npk_concrete"],
                "NPK_Formwork": npk["npk_formwork"],
                "NPK_Label": npk["npk_label"],
                "Height_Category": npk["height_category"],
                "Notes": npk["notes"],
            })
        else:
            row.update({
                "NPK_Concrete": None,
                "NPK_Formwork": None,
                "NPK_Label": None,
                "Height_Category": None,
                "Notes": "Missing Betontyp or Height",
            })
        
        wall_rows.append(row)
    
    df_walls = pd.DataFrame(wall_rows)
    
    # =========== EXTRACT SLABS ===========
    slabs = model.by_type("IfcSlab")
    slab_rows = []
    
    for slab in slabs:
        psets = get_psets(slab)
        qto = psets.get("Qto_SlabBaseQuantities", {})
        
        betontyp = extract_betontyp(psets)
        
        # Try multiple methods to get thickness (priority: wg_SUB > QTO > name > calculated)
        thickness = extract_slab_thickness(slab, psets, qto)
        
        row = {
            "GlobalId": slab.GlobalId,
            "Name": slab.Name or "",
            "Betontyp": betontyp,
            "Thickness_m": thickness,
            "NetVolume_m3": qto.get("NetVolume"),
            "GrossArea_m2": qto.get("GrossArea"),
            "Perimeter_m": qto.get("Perimeter"),
        }
        
        if betontyp and thickness:
            npk = generate_slab_positions(betontyp, thickness)
            row.update({
                "NPK_Concrete": npk["npk_concrete"],
                "NPK_Formwork": npk["npk_formwork"],
                "NPK_Label": npk["npk_label"],
                "Thickness_Category": npk["thickness_category"],
                "Notes": npk["notes"],
            })
        else:
            row.update({
                "NPK_Concrete": None,
                "NPK_Formwork": None,
                "NPK_Label": None,
                "Thickness_Category": None,
                "Notes": "Missing Betontyp or Thickness",
            })
        
        slab_rows.append(row)
    
    df_slabs = pd.DataFrame(slab_rows)
    
    # =========== SUMMARY ===========
    print(f"\n✅ IFC model loaded successfully!")
    print(f"🏗️ Schema: {model.schema}")
    print(f"\n📊 WALLS: {len(df_walls)} elements")
    if len(df_walls) > 0:
        valid_walls = df_walls["NPK_Concrete"].notna().sum()
        print(f"   ├─ With NPK position: {valid_walls}")
        print(f"   └─ Missing data: {len(df_walls) - valid_walls}")
    
    print(f"\n📊 SLABS: {len(df_slabs)} elements")
    if len(df_slabs) > 0:
        valid_slabs = df_slabs["NPK_Concrete"].notna().sum()
        print(f"   ├─ With NPK position: {valid_slabs}")
        print(f"   └─ Missing data: {len(df_slabs) - valid_slabs}")
    
    # Show sample data
    if len(df_walls) > 0:
        print("\n📋 Sample wall data:")
        display(df_walls[["GlobalId", "Betontyp", "Height_m", "NPK_Concrete", "NPK_Formwork", "Height_Category"]].head())


def on_load_clicked(b):
    """Handle load button click."""
    clear_output(wait=True)
    display(loading_widget)
    
    if loading_method.value == 'A':
        if github_dropdown.value and 'Error' not in github_dropdown.value:
            import urllib.request
            url = f'https://raw.githubusercontent.com/{GITHUB_REPO_OWNER}/{GITHUB_REPO_NAME}/main/{GITHUB_FOLDER_PATH}/{github_dropdown.value}'
            local_file = github_dropdown.value
            
            print(f"📥 Downloading {github_dropdown.value}...")
            try:
                urllib.request.urlretrieve(url, local_file)
                load_and_process_ifc(local_file)
            except Exception as e:
                print(f"❌ Download error: {e}")
    
    elif loading_method.value == 'B':
        if file_upload.value:
            uploaded = next(iter(file_upload.value.values()))
            filename = uploaded['metadata']['name']
            with open(filename, 'wb') as f:
                f.write(uploaded['content'])
            load_and_process_ifc(filename)
        else:
            print("❌ Please upload a file first.")


# Create UI
print("🔧 IFC Model Loading")

loading_method = widgets.RadioButtons(
    options=[('A) GitHub Repository', 'A'), ('B) Upload File', 'B')],
    value='A',
    description='Source:'
)

github_files = fetch_github_files()
github_dropdown = widgets.Dropdown(
    options=github_files,
    value=github_files[0] if github_files else None,
    description='File:',
    layout=widgets.Layout(width='500px')
)

file_upload = widgets.FileUpload(accept='.ifc', multiple=False, description='Upload:')

load_button = widgets.Button(description='Load Model', button_style='success', icon='download')
load_button.on_click(on_load_clicked)

loading_widget = widgets.VBox([
    loading_method,
    github_dropdown,
    file_upload,
    load_button
])

display(loading_widget)

# Auto-load default
if github_files and 'Error' not in github_files[0]:
    print("\n🔄 Auto-loading default file...")
    import urllib.request
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO_OWNER}/{GITHUB_REPO_NAME}/main/{GITHUB_FOLDER_PATH}/{github_files[0]}'
    try:
        urllib.request.urlretrieve(url, github_files[0])
        load_and_process_ifc(github_files[0])
    except Exception as e:
        print(f"⚠️ Auto-load failed: {e}")

## 5. Data Quality Check

Review data quality before generating the final report.

In [ ]:
def analyze_data_quality(df: pd.DataFrame, element_type: str) -> None:
    """Print data quality summary for a DataFrame."""
    print(f"\n{'='*50}")
    print(f"📊 {element_type.upper()} DATA QUALITY")
    print(f"{'='*50}")
    
    total = len(df)
    if total == 0:
        print("No elements found.")
        return
    
    # NPK coverage
    with_npk = df["NPK_Concrete"].notna().sum()
    aufmass = (df["NPK_Concrete"] == "AUFMASS").sum()
    missing = total - with_npk
    
    print(f"\nTotal elements: {total}")
    print(f"✅ Valid NPK positions: {with_npk - aufmass} ({(with_npk-aufmass)/total*100:.1f}%)")
    print(f"⚠️  Requires Aufmass: {aufmass} ({aufmass/total*100:.1f}%)")
    print(f"❌ Missing data: {missing} ({missing/total*100:.1f}%)")
    
    # Betontyp distribution
    print(f"\nBetontyp distribution:")
    betontyp_counts = df["Betontyp"].value_counts(dropna=False)
    for bt, count in betontyp_counts.items():
        label = bt if pd.notna(bt) else "(missing)"
        npk_info = BETONTYP_TO_NPK.get(str(bt), ("unknown", 0))
        print(f"   {label}: {count} -> {npk_info[0] if bt else 'N/A'}")
    
    # Issues breakdown
    if "Notes" in df.columns:
        issues = df[df["Notes"].notna()]["Notes"].value_counts()
        if len(issues) > 0:
            print(f"\n⚠️  Issues found:")
            for issue, count in issues.head(5).items():
                print(f"   {issue}: {count}")


# Run quality check
if df_walls is not None:
    analyze_data_quality(df_walls, "Walls")

if df_slabs is not None:
    analyze_data_quality(df_slabs, "Slabs")

## 6. Generate NPK Summary

Create aggregated summaries by NPK position for cost estimation.

In [ ]:
def create_npk_summary(df: pd.DataFrame, category_col: str) -> pd.DataFrame:
    """Create NPK summary grouped by position and category."""
    # Filter to valid positions
    valid = df[(df["NPK_Concrete"].notna()) & (df["NPK_Concrete"] != "AUFMASS")].copy()
    
    if len(valid) == 0:
        print("⚠️ No valid NPK positions for summary")
        return pd.DataFrame()
    
    # Aggregate
    summary = valid.groupby(["NPK_Concrete", "NPK_Formwork", "NPK_Label", category_col]).agg(
        Count=("GlobalId", "count"),
        Total_Volume_m3=("NetVolume_m3", "sum"),
    ).reset_index()
    
    # Round volumes
    summary["Total_Volume_m3"] = summary["Total_Volume_m3"].round(2)
    
    return summary.sort_values("NPK_Concrete")


# Create summaries
if df_walls is not None and len(df_walls) > 0:
    df_walls_summary = create_npk_summary(df_walls, "Height_Category")
    print("\n📊 WALL NPK SUMMARY:")
    display(df_walls_summary)

if df_slabs is not None and len(df_slabs) > 0:
    df_slabs_summary = create_npk_summary(df_slabs, "Thickness_Category")
    print("\n📊 SLAB NPK SUMMARY:")
    display(df_slabs_summary)

## 7. Export to Excel

Generate professional Excel output with:
- Auto-filters on all columns
- Dynamic SUBTOTAL formulas
- Formatted headers

In [ ]:
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill

def export_npk_excel(output_path: str):
    """Export all NPK data to formatted Excel file."""
    
    HEADER_FILL = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    HEADER_FONT = Font(bold=True, color="FFFFFF")
    
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        
        def write_sheet(df, sheet_name, numeric_cols):
            """Write DataFrame to sheet with formatting."""
            if df is None or len(df) == 0:
                return
            
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            ws = writer.sheets[sheet_name]
            
            # Format headers
            for col_num in range(1, len(df.columns) + 1):
                cell = ws.cell(row=1, column=col_num)
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
            
            # Auto-filter
            max_row = len(df) + 1
            max_col = get_column_letter(len(df.columns))
            ws.auto_filter.ref = f"A1:{max_col}{max_row}"
            
            # Total row with SUBTOTAL
            total_row = max_row + 1
            ws.cell(row=total_row, column=1, value="TOTAL").font = Font(bold=True)
            
            for col_name in numeric_cols:
                if col_name in df.columns:
                    col_idx = df.columns.get_loc(col_name) + 1
                    col_letter = get_column_letter(col_idx)
                    formula = f"=SUBTOTAL(109,{col_letter}2:{col_letter}{max_row})"
                    ws.cell(row=total_row, column=col_idx, value=formula).font = Font(bold=True)
        
        # Write sheets
        write_sheet(df_walls, "Walls_Detail", ["NetVolume_m3", "NetSideArea_m2"])
        write_sheet(df_walls_summary, "Walls_NPK_Summary", ["Count", "Total_Volume_m3"])
        write_sheet(df_slabs, "Slabs_Detail", ["NetVolume_m3", "GrossArea_m2"])
        write_sheet(df_slabs_summary, "Slabs_NPK_Summary", ["Count", "Total_Volume_m3"])
    
    print(f"✅ Excel exported: {output_path}")
    print(f"   Sheets: Walls_Detail, Walls_NPK_Summary, Slabs_Detail, Slabs_NPK_Summary")


# Export
OUTPUT_FILE = "NPK_241_Auswertung.xlsx"
export_npk_excel(OUTPUT_FILE)

# Download link for Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print(f"\n📁 File saved to: {OUTPUT_FILE}")

## 8. Next Steps

**Extend the analysis:**
- Add more element types (IfcColumn, IfcBeam)
- Include edge formwork positions (268.xxx)
- Calculate formwork areas from geometry

**Improve data quality:**
- Handle elements >4.0m with Aufmass positions
- Add validation for impossible combinations
- Cross-reference with eBKP classifications

**Try different models:**
- Upload your own IFC file using Option B
- Compare different project phases
- Validate NPK positions against manual takeoffs